In [1]:
%run -i llm4ad_snippets/__init__.py
%run -i llm4ad_snippets/code.py

In [2]:
import dataclasses
from typing import Optional, List, Dict

In [58]:
import uuid


In [66]:
uuid.uuid1()

UUID('7ed6070a-12f7-11f0-bbb9-9457a5f25bd5')

In [3]:

@dataclasses.dataclass
class Evaluation:
    fitness: float
    errors: Optional[str]

class Candidate:
    def __init__(self, mycode, evaluation: Optional[Evaluation] = None, auto_eval=True):
        if type(mycode) is str:
            self.program = TextFunctionProgramConverter.text_to_program(mycode)
        elif type(mycode) is Program:
            self.program = mycode
        elif type(mycode) is Candidate:
            self.program = mycode.program
            self.evaluation = mycode.evaluation
        else:
            raise Exception(f'unknown type {type(mycode)} passed to `Candidate` constructor')
        self.evaluation = evaluation

    def evaluate(self, evaluator_func: callable, func_to_eval: str):
        self.evaluation = evaluator_func([ff for ff in self.program.exec() if ff.__name__ == func_to_eval][0])
        return self


class Generation:
    def __init__(self, initial_cands=[]):
        self._candidates = list(map(Candidate, initial_cands))

    def add(self, *prog):
        self._candidates.extend(prog)

    def sort_by(self, key_func):
        self._candidates = list(sorted(self._candidates, key=key_func, reverse=True))

    def evaluate_all(self, evaluator_func, func_to_eval: str):
        for idx, cand in enumerate(self._candidates):
            if cand.evaluation is None:
                self._candidates[idx].evaluate(evaluator_func, func_to_eval)
    

In [5]:

task_prompt = "Write an algorithm that uses a seed integer and some custom formula to calcuate a new pseudorandom number."
template = """
def pseudorandom_number(seed: int) -> int:
    " use the seed value to generate a pseudorandom number somehow "
"""
template_target = "pseudorandom_number"

INITIAL_PROMPT = f'''{task_prompt}
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed {{}}. 
2. Next, implement the following Python function:
{str(template)}
Do not give additional explanations.'''

def build_initial_prompt():
    INITIAL_PROMPT = f'''{task_prompt}
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed {{}}. 
2. Next, implement the following Python function:
{str(template)}
Do not give additional explanations.'''

def build_mutate_prompt(parent_prompt):
    MUTATE_PROMPT = f'''{task_prompt}
I have one algorithm with its code as follows.
Code:
{str(parent_prompt)}
Please assist me in creating a new algorithm that has a different form but can be a modified version of the algorithm provided.
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed {{}}.
2. Next, implement the following Python function:
{str(template)}
Do not give additional explanations.'''
    return MUTATE_PROMPT

def mutation_prompt(parent_cand):
    parent_implementation = str(parent_cand.program)
    return build_mutate_prompt(parent_implementation)

In [6]:
import os

def draw_sample_program(prompt, template=template):
    llm_sampler = SampleTrimmer(HttpsApi(
    host="api.openai.com",
    key=os.environ['OPENAI_API_KEY'],  # your key, e.g., sk-xxxxxxxxxx
    model="gpt-4o-mini",  # your llm, e.g., gpt-3.5-turbo, deepseek-chat
    timeout=20
))
    return llm_sampler.sample_to_program(llm_sampler.draw_sample(prompt), template)

def draw_sample_candidate(prompt, template=template):
    return Candidate(draw_sample_program(prompt, template))


In [49]:
class SamplerEngine:
    def __init__(self, *args, **kwargs):
        self._sampler = HttpsApi(*args, **kwargs)
        self._trimmer = SampleTrimmer(self._sampler)

    def sample_program(self, prompt, template):
        return self._trimmer.sample_to_program(self._trimmer.draw_sample(prompt), template)

    def _candidate_from_prompt(self, prompt_builder_func, template):
        def _sampling_func(*args, **kwargs):
            resa = prompt_builder_func(*args, **kwargs)
            res = self.sample_program(resa, template)
            return Candidate(res, evaluation=None)
        return _sampling_func
    
    def candidate_from_prompt(self, template):
        def _prompt_dec(_func):
            return self._candidate_from_prompt(_func, template)
        return _prompt_dec
            

In [50]:
eng = SamplerEngine(
    host="api.openai.com",
    key=os.environ['OPENAI_API_KEY'],  # your key, e.g., sk-xxxxxxxxxx
    model="gpt-4o-mini",  # your llm, e.g., gpt-3.5-turbo, deepseek-chat
    timeout=20
)

In [51]:
@eng.candidate_from_prompt(template)
def build_initial_prompt():
    return f'''{task_prompt}
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed {{}}. 
2. Next, implement the following Python function:
{str(template)}
Do not give additional explanations.'''


In [52]:
xx = build_initial_prompt()
xx

/vol/bitbucket/jpk24/thesis/reallm/llm4ad_snippets/code.py:187: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node.body[0], ast.Expr) and isinstance(node.body[0].value, ast.Str):


In [54]:
xx.program

Program(preface='', functions=[Function(name='pseudorandom_number', args='seed: int', body='    return (seed * 48271 + 1) % 2147483647', return_type='int', docstring=' use the seed value to generate a pseudorandom number somehow ', score=None, evaluate_time=None, sample_time=None)])

In [55]:
xx.evaluation

In [67]:
import random
import numpy as np

def evaluate_pseudorandom(func):
    randseed = random.randint(1,23489)
    try:
        res = func(randseed)
        return Evaluation(
            fitness = res,
            errors = None
        )
    except Exception as e:
        return Evaluation(
            fitness = float('-inf'),
            errors = str(e)
        )

## Sane Main Loop

In [124]:
max_generations=2
max_population=4
k=2
num_mutations=3

all_evaluations = []

current_gen = Generation()
current_gen.add(draw_sample_candidate(INITIAL_PROMPT).evaluate(evaluate_pseudorandom, 'pseudorandom_number'))

all_evaluations.append({'idx': 0, 'fitness': current_gen._candidates[0].evaluation.fitness})
_cidx = 1

for g_idx in range(max_generations):
    # order by objective
    current_gen.sort_by(lambda cand: cand.evaluation.fitness)
    ## select top K
    topk_cands = current_gen._candidates[:k]
    
    ## mutate top K
    mutated_cands = []
    for elite_cand in topk_cands:
        for _ in range(num_mutations):
            new_cand = draw_sample_candidate(mutation_prompt(elite_cand)).evaluate(evaluate_pseudorandom, 'pseudorandom_number')
            all_evaluations.append({'cand_idx': _cidx, 'fitness': new_cand.evaluation.fitness})
            _cidx+=1
            mutated_cands.append(
                new_cand
            )
    
    # add to new generation
    current_gen.add(*mutated_cands)
    current_gen.sort_by(lambda cand: cand.evaluation.fitness)
    
    # dropout
    current_gen._candidates = current_gen._candidates[:min(max_population, len(current_gen._candidates))]


/vol/bitbucket/jpk24/thesis/reallm/llm4ad_snippets/code.py:187: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node.body[0], ast.Expr) and isinstance(node.body[0].value, ast.Str):


In [125]:
all_evaluations

[{'idx': 0, 'fitness': 4256771197},
 {'cand_idx': 1, 'fitness': 1396076031},
 {'cand_idx': 2, 'fitness': 1301817732},
 {'cand_idx': 3, 'fitness': 1350143739},
 {'cand_idx': 4, 'fitness': 588763244},
 {'cand_idx': 5, 'fitness': 2260024797},
 {'cand_idx': 6, 'fitness': 171535578},
 {'cand_idx': 7, 'fitness': 2293916148},
 {'cand_idx': 8, 'fitness': 1006459649},
 {'cand_idx': 9, 'fitness': 4145562045}]

In [126]:
current_gen._candidates

In [103]:
topk_cands

In [90]:
current_gen._candidates[0].evaluation

Evaluation(fitness=1067368353, errors=None)